# 07 · Models that depend on normalisation (k-NN, SVM, naive Bayes, neural network)

These models measure distances or assume feature scales, so they show most clearly *why* normalisation matters.
All use FS_TRANSFER. The neural network is also a check of the benchmarking doc's warning that flexible models
need roughly ten times more data (learning curves in notebook 09).

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "07"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
KNN = lambda: KNeighborsClassifier(n_neighbors=50, weights="distance")
for rid, kind in [("OT-01a", "none"), ("OT-01b", "z"), ("OT-01c", "log_z"), ("OT-01d", "quantile")]:
    run(rid, M.pipeline(M.preprocess(kind), KNN()), "FS_TRANSFER", f"k-NN (k=50) · {M.PREPROCESSORS[kind]}")
run("OT-01e", M.tuned(M.pipeline(M.preprocess("quantile"), KNeighborsClassifier(weights="distance")), {"kneighborsclassifier__n_neighbors": randint(5, 200)}),
    "FS_TRANSFER", "k-NN, k tuned · quantile")
run("OT-02", M.pipeline(M.preprocess("log_z"), CalibratedClassifierCV(SVC(kernel="rbf", C=1.0), method="sigmoid", cv=3)), "FS_TRANSFER", "SVM (RBF) + Platt calibration · log1p + z")
run("OT-03", M.pipeline(M.preprocess("quantile"), GaussianNB()), "FS_TRANSFER", "Gaussian naive Bayes · quantile")
MLP = lambda: MLPClassifier(hidden_layer_sizes=(32,), early_stopping=True, max_iter=1000, random_state=M.SEED)
run("OT-04", M.pipeline(M.preprocess("log_z"), MLP()), "FS_TRANSFER", "Neural network (1 hidden layer, 32), early stopping · log1p + z")
run("OT-04b", M.tuned(M.pipeline(M.preprocess("log_z"), MLP()), {"mlpclassifier__alpha": loguniform(1e-4, 10),
    "mlpclassifier__hidden_layer_sizes": [(8,), (16,), (32,), (16, 8)]}), "FS_TRANSFER", "Neural network, size and L2 tuned");

In [ ]:
board = M.leaderboard(notebook=NB)
cols = ["run_id", "description", "feature_set", "auc", "auc_sd", "brier", "cal_slope", "top20", "n_features", "fit_s"]
board[cols]

In [ ]:
r = M.read_results(); r = r[r.notebook == NB]
plt.figure(figsize=(10, 0.35 * r.run_id.nunique() + 1.2))
sns.boxplot(data=r, y="run_id", x="auc", order=board.run_id, color="#8DA0CB")
plt.axvline(M.leaderboard().query("run_id == 'REF-LR'").auc.iloc[0], ls="--", c="#C44E52", label="REF-LR (notebook 01)")
plt.legend(); plt.title("AUC across the same 25 folds"); plt.ylabel(""); plt.show()

## Normalisation effect on k-NN

In [ ]:
k = board[board.run_id.str.startswith("OT-01")].set_index("run_id")[["description", "auc", "auc_sd"]]
k.sort_index()

## Findings (Salud-tuned world)
- **Scaling matters for k-NN:**

  | Scaling | AUC |
  |---|---|
  | raw | 0.802 |
  | z-score | 0.843 |
  | quantile | 0.841 |
  | **log1p + z** | **0.863** |

  Distance models need the log step, as the EDA predicted.
- SVM with Platt calibration (0.872) is level with the scorecard. The MLP (0.859) and naive Bayes (0.833, very poorly calibrated) add nothing.